Ссылка на урок: [Спринт 2/5: 2 RAG: от сырых данных до точных ответов → Тема 3/4: Архитектура RAG → Урок 6/6: Сборка RAG-пайплайна с LangChain]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/b36d9f5f-3f26-4fbb-bd65-71d33c4523ef/lessons/827f1ac2-fe38-474d-843b-7f4ea53670d3/#5cdb0809-e9b3-4dea-925a-a61b33f30f2b )

In [2]:
from collections.abc import Iterator

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document
from pypdf import PdfReader


class PypdfPageLoader(BaseLoader):
    """Читает PDF постранично и возвращает объекты Document."""

    def __init__(self, file_path: str):
        self.file_path = file_path

    def lazy_load(self) -> Iterator[Document]:
        """Читает страницы лениво, сохраняя источник и номер страницы."""
        reader = PdfReader(self.file_path)
        file_metadata = {
            key.removeprefix("/").lower(): value
            for key, value in (reader.metadata or {}).items()
        }

        for page_number, page in enumerate(reader.pages):
            yield Document(
                page_content=page.extract_text() or "",
                metadata={
                    **file_metadata,
                    "source": self.file_path,
                    "page": page_number,
                    "total_pages": len(reader.pages),
                },
            )


loader = PypdfPageLoader("my_doc.pdf")
docs = loader.load()

print(len(docs))                    # 3 — по одному Document на страницу PDF
print(docs[0].metadata["source"])   # my_doc.pdf
print(docs[0].metadata["page"])     # 0 — нумерация страниц с нуля

FileNotFoundError: [Errno 2] No such file or directory: 'my_doc.pdf'

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
chunks = splitter.split_documents(docs)

print(len(chunks))          # 9 чанков из 3 страниц

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True, "prompt": "passage: "},
    query_encode_kwargs={"normalize_embeddings": True, "prompt": "query: "},
)

In [ ]:
import numpy as np

v_doc = np.array(embeddings.embed_documents(["тест"])[0])
v_query = np.array(embeddings.embed_query("тест"))
print(round(float(v_doc @ v_query), 4))    # ~0.93, но не 1.0 — префиксы на месте

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

found = retriever.invoke("что означает код ошибки E-52")
print([d.metadata["page"] for d in found])    # Страницы найденных чанков

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="http://localhost:8080/v1",
    api_key="not-needed",
    model="qwen3.5",
    temperature=0.0,
    max_tokens=200,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — ассистент поддержки. Отвечай ТОЛЬКО по приведённым фрагментам "
     "базы знаний. Если ответа в них нет, ответь: «В базе знаний ответа "
     "нет». Отвечай кратко, одним-двумя предложениями."),
    ("human", "Фрагменты базы знаний:\n{context}\n\nВопрос: {question}"),
])


def format_docs(docs):
    """Склеивает найденные Document в нумерованный контекст — как
    build_messages() в прошлом уроке."""
    return "\n".join(f"[{n}] {d.page_content}" for n, d in enumerate(docs, 1))


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("что означает код ошибки E-52")